# 逐步理解 `bpe_openai_gpt2.py`

这个 Notebook 用来配合阅读同目录下的 `bpe_openai_gpt2.py`。

目标不是重新训练 tokenizer，而是看懂 GPT-2 原始 BPE tokenizer 的几个关键步骤：

- 字节到 Unicode 字符的可逆映射
- 正则表达式如何先粗分文本
- BPE 如何反复合并相邻符号
- `encode` 如何把文本变成 token ID
- `decode` 如何把 token ID 还原成文本


## 0. 准备环境

下面这个单元格只做两件事：

- 找到 `bpe_openai_gpt2.py` 所在目录。
- 把这个目录加入 `sys.path`，方便后面的单元格导入代码。

它不会切换当前工作目录，因此在 VS Code/Jupyter 里更稳定。


In [ ]:
from pathlib import Path
import sys

if Path("bpe_openai_gpt2.py").exists():
    BPE_DIR = Path.cwd()
else:
    BPE_DIR = Path.cwd() / "ch02" / "02_bonus_bytepair-encoder"

BPE_DIR = BPE_DIR.resolve()

if not (BPE_DIR / "bpe_openai_gpt2.py").exists():
    raise FileNotFoundError(f"没有找到 bpe_openai_gpt2.py，请确认当前项目路径是否正确: {BPE_DIR}")

if str(BPE_DIR) not in sys.path:
    sys.path.insert(0, str(BPE_DIR))

print("当前工作目录:", Path.cwd())
print("BPE 代码目录:", BPE_DIR)
print("是否找到 bpe_openai_gpt2.py:", (BPE_DIR / "bpe_openai_gpt2.py").exists())


In [ ]:
import inspect
import regex as re

from bpe_openai_gpt2 import (
    bytes_to_unicode,
    get_pairs,
    Encoder,
    download_vocab,
    get_encoder,
)

## 1. 文件整体结构

`bpe_openai_gpt2.py` 主要由这些部分组成：

- `bytes_to_unicode()`：构造字节到 Unicode 字符的映射表。
- `get_pairs(word)`：找出一个 token 内部所有相邻符号对。
- `Encoder`：真正负责 BPE 编码和解码的类。
- `get_encoder(model_name, models_dir)`：从 `encoder.json` 和 `vocab.bpe` 加载 GPT-2 tokenizer。
- `download_vocab()`：下载 GPT-2 的 tokenizer 词表文件。

先看一下这两个辅助函数的源码。


In [ ]:
print(inspect.getsource(bytes_to_unicode))
print(inspect.getsource(get_pairs))

## 2. `bytes_to_unicode()`：为什么需要字节映射？

GPT-2 的 BPE 不是直接在 Python 字符上做合并，而是先把文本编码成 UTF-8 字节，再把每个字节映射成一个 Unicode 字符。

这样做的好处是：

- 任意文本最终都能拆成字节，因此不会真的遇到无法表示的字符。
- tokenizer 可以做到可逆：编码后再解码，能回到原文本。
- 避免把原始空白字符、控制字符直接交给 BPE 处理。

下面看几个字节会映射到什么字符。


In [ ]:
byte_encoder = bytes_to_unicode()

for b in [65, 32, 10, 33, 128, 255]:
    mapped = byte_encoder[b]
    print(f"byte {b:>3} -> {mapped!r} (Unicode code point: {ord(mapped)})")

注意：空格字节 `32` 不会映射成普通空格，而是映射成另一个 Unicode 字符。

这是 GPT-2 tokenizer 中经常看到 `Ġ` 这类字符的原因之一：它们通常表示原文本里带有前导空格的 token。


In [ ]:
text = " Hello"
byte_encoded = "".join(byte_encoder[b] for b in text.encode("utf-8"))

print("原文本:", repr(text))
print("UTF-8 字节:", list(text.encode("utf-8")))
print("字节映射后的字符串:", repr(byte_encoded))

## 3. `get_pairs(word)`：找相邻符号对

BPE 的核心动作是“合并一对相邻符号”。

所以每一步都要先找出当前 token 里有哪些相邻符号对。


In [ ]:
word = tuple("lower")
pairs = get_pairs(word)

print("word:", word)
print("pairs:", pairs)

如果当前符号序列是：

```text
('l', 'o', 'w', 'e', 'r')
```

那么相邻符号对就是：

```text
('l', 'o'), ('o', 'w'), ('w', 'e'), ('e', 'r')
```

BPE 会查 `vocab.bpe` 里的合并优先级，选择优先级最高的一对来合并。


## 4. `Encoder.__init__`：初始化时保存了什么？

`Encoder` 初始化时会保存几张关键表：

- `encoder`：字符串 token 到整数 ID 的映射。
- `decoder`：整数 ID 到字符串 token 的反向映射。
- `byte_encoder`：字节到 Unicode 字符的映射。
- `byte_decoder`：Unicode 字符到字节的反向映射。
- `bpe_ranks`：BPE 合并规则到优先级排名的映射。
- `cache`：缓存已经处理过的 token，避免重复计算。

再看一下 `Encoder` 的三个核心方法源码。


In [ ]:
print(inspect.getsource(Encoder.bpe))
print(inspect.getsource(Encoder.encode))
print(inspect.getsource(Encoder.decode))

## 5. 正则表达式：先把文本粗分成片段

`encode()` 并不是一开始就对整段文本做 BPE。

它先用下面这个正则表达式把文本粗略切成片段：

```python
's|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+
```

直观理解：

- 英文缩写如 `'s`、`'t`、`'re` 单独处理。
- 字母序列、数字序列、标点序列分开处理。
- 前导空格可能会和后面的词放在同一个片段里。


In [ ]:
dummy_encoder = Encoder(encoder={}, bpe_merges=[])

sample = "Hello, world! It's 2026."
parts = re.findall(dummy_encoder.pat, sample)

print(parts)

model_dir = BPE_DIR / "gpt2_model"

if not (model_dir / "encoder.json").exists() or not (model_dir / "vocab.bpe").exists():
    import os

    old_cwd = Path.cwd()
    try:
        os.chdir(BPE_DIR)
        download_vocab()
    finally:
        os.chdir(old_cwd)

tokenizer = get_encoder("gpt2_model", str(BPE_DIR))

print("词表大小:", len(tokenizer.encoder))
print("BPE 合并规则数量:", len(tokenizer.bpe_ranks))
print("前 10 个 token ID:")
for i, item in enumerate(tokenizer.encoder.items()):
    print(item)
    if i >= 9:
        break


In [ ]:
model_dir = Path("gpt2_model")

if not (model_dir / "encoder.json").exists() or not (model_dir / "vocab.bpe").exists():
    download_vocab()

tokenizer = get_encoder("gpt2_model", ".")

print("词表大小:", len(tokenizer.encoder))
print("BPE 合并规则数量:", len(tokenizer.bpe_ranks))
print("前 10 个 token ID:")
for i, item in enumerate(tokenizer.encoder.items()):
    print(item)
    if i >= 9:
        break

## 7. `encode()`：文本到 token ID

`encode()` 的流程可以概括为：

```text
原始文本
-> 正则表达式粗分
-> 每个片段转 UTF-8 字节
-> 字节映射到 Unicode 字符
-> 对这个字符序列执行 BPE 合并
-> 每个 BPE token 查表得到整数 ID
```


In [ ]:
text = "Hello, world! This is a tokenizer test."
ids = tokenizer.encode(text)

print("原文本:", text)
print("token IDs:", ids)
print("token 数量:", len(ids))

下面把 `encode()` 里的几个中间步骤拆开看。


In [ ]:
parts = re.findall(tokenizer.pat, text)
print("正则粗分结果:")
print(parts)

print("\n每个片段的字节映射与 BPE 结果:")
for part in parts:
    byte_token = "".join(tokenizer.byte_encoder[b] for b in part.encode("utf-8"))
    bpe_result = tokenizer.bpe(byte_token)
    bpe_pieces = bpe_result.split(" ")
    bpe_ids = [tokenizer.encoder[piece] for piece in bpe_pieces]
    print(f"{part!r:>12} -> {byte_token!r:>16} -> {bpe_pieces} -> {bpe_ids}")

## 8. 跟踪一次 BPE 合并过程

下面这个辅助函数不会改变原文件，只是把 `Encoder.bpe()` 的核心循环打印出来，帮助观察每一步合并了哪一对符号。


In [ ]:
def trace_bpe(tokenizer, text_piece, max_steps=20):
    byte_token = "".join(tokenizer.byte_encoder[b] for b in text_piece.encode("utf-8"))
    word = tuple(byte_token)
    pairs = get_pairs(word)

    print("文本片段:", repr(text_piece))
    print("字节映射:", repr(byte_token))
    print("初始符号:", word)

    step = 0
    while pairs and step < max_steps:
        bigram = min(pairs, key=lambda pair: tokenizer.bpe_ranks.get(pair, float("inf")))
        rank = tokenizer.bpe_ranks.get(bigram)
        if rank is None:
            print("没有可合并的符号对，停止。")
            break

        first, second = bigram
        print(f"step {step}: 合并 {bigram}，rank={rank}")

        new_word = []
        i = 0
        while i < len(word):
            try:
                j = word.index(first, i)
                new_word.extend(word[i:j])
                i = j
            except ValueError:
                new_word.extend(word[i:])
                break

            if word[i] == first and i < len(word) - 1 and word[i + 1] == second:
                new_word.append(first + second)
                i += 2
            else:
                new_word.append(word[i])
                i += 1

        word = tuple(new_word)
        print("       ->", word)

        if len(word) == 1:
            break
        pairs = get_pairs(word)
        step += 1

    print("最终 BPE token 字符串:", " ".join(word))
    return word

In [ ]:
trace_bpe(tokenizer, " tokenizer")

## 9. `decode()`：token ID 回到文本

`decode()` 做的是 `encode()` 的反向过程：

```text
token ID
-> 查 decoder 得到 BPE token 字符串
-> 拼成一个字节映射后的字符串
-> 通过 byte_decoder 还原成原始 UTF-8 字节
-> 解码成普通文本
```


In [ ]:
decoded = tokenizer.decode(ids)

print("原文本:  ", repr(text))
print("解码结果:", repr(decoded))
print("是否完全一致:", decoded == text)

## 10. 和 `tiktoken` 对照

这个原始实现和 `tiktoken` 的 GPT-2 编码结果通常应该一致。

区别在于：

- 这个文件是更容易阅读的 Python 原始实现。
- `tiktoken` 是更快的工程实现，核心部分用 Rust 编写。
- `tiktoken.encode()` 支持 `allowed_special` 等额外参数；本文件里的 `Encoder.encode(self, text)` 没有这些参数。


In [ ]:
import tiktoken

tiktoken_tokenizer = tiktoken.get_encoding("gpt2")
tiktoken_ids = tiktoken_tokenizer.encode(text)

print("原始实现:", ids)
print("tiktoken:", tiktoken_ids)
print("是否一致:", ids == tiktoken_ids)

## 11. 小练习

你可以修改下面的 `text`，观察：

- 英文单词前面有没有空格，会不会影响 token ID？
- 标点符号是否单独成 token？
- 中文、emoji 这类字符会被拆成多少个 token？


In [ ]:
texts = [
    "hello",
    " hello",
    "Hello, world!",
    "我喜欢学习 tokenizer",
    "GPT-2 BPE 😊",
]

for text in texts:
    ids = tokenizer.encode(text)
    print("=" * 60)
    print("文本:", repr(text))
    print("token IDs:", ids)
    print("token 数量:", len(ids))
    print("decode:", repr(tokenizer.decode(ids)))

## 总结

`bpe_openai_gpt2.py` 的核心可以记成一句话：

```text
先用正则把文本切成片段，再把片段转成字节安全的 Unicode 字符串，然后按 GPT-2 训练好的 BPE 合并规则反复合并，最后查表得到 token ID。
```

最重要的函数关系：

```text
bytes_to_unicode() 负责字节安全映射
get_pairs()        负责找可合并的相邻符号对
Encoder.bpe()      负责执行 BPE 合并
Encoder.encode()   负责文本 -> token ID
Encoder.decode()   负责 token ID -> 文本
```
